# Phase 0: data and tools

Checks that the MIDI-B validation download is complete and readable, that every answer-key row matches a file,
that the mapping tables cover all patients and UIDs, and picks the 20-patient dev-mini subset.

In [1]:
import os
import sys
import json
import glob
import sqlite3
import platform

import numpy as np
import pandas as pd
import pydicom

from fpdeid import paths
from fpdeid import dicom_io

os.makedirs(paths.DERIVED, exist_ok=True)
phase_results = os.path.join(paths.RESULTS, "phase0")
os.makedirs(phase_results, exist_ok=True)

## 1. Environment

In [2]:
print("python  :", platform.python_version())
print("pydicom :", pydicom.__version__)
print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("CPUs    :", os.cpu_count())

python  : 3.10.12
pydicom : 3.0.2
pandas  : 2.3.3
numpy   : 2.2.6
CPUs    : 48


## 2. What TCIA says vs what I downloaded

In [3]:
with open(paths.VALIDATION_SERIES_LIST) as f:
    series_list = json.load(f)

tcia = pd.DataFrame(series_list)
tcia["ImageCount"] = tcia["ImageCount"].astype(int)
tcia["FileSize"] = tcia["FileSize"].astype(float)

print("series  :", len(tcia))
print("patients:", tcia["PatientID"].nunique())
print("studies :", tcia["StudyInstanceUID"].nunique())
print("images  :", tcia["ImageCount"].sum())
total_gb = tcia["FileSize"].sum() / 1e9
print("size GB :", round(total_gb, 2))

series  : 280
patients: 216
studies : 241
images  : 23921
size GB : 7.63


In [4]:
# count .dcm files that are really on disk, per series folder
on_disk = []
for series in series_list:
    folder = os.path.join(paths.VALIDATION_IMAGES, series["PatientID"], series["SeriesInstanceUID"])
    pattern = os.path.join(folder, "*.dcm")
    n_files = len(glob.glob(pattern))
    on_disk.append(n_files)

tcia["files_on_disk"] = on_disk
tcia["complete"] = tcia["files_on_disk"] == tcia["ImageCount"]

expected = tcia["ImageCount"].sum()
found = tcia["files_on_disk"].sum()
print("files expected:", expected)
print("files found   :", found)
print("completeness  :", round(100 * found / expected, 2), "%")
print("incomplete series:", (~tcia["complete"]).sum())

files expected: 23921
files found   : 23921
completeness  : 100.0 %
incomplete series: 0


### Inventory per modality

In [5]:
inventory = tcia.groupby("Modality").agg(
    patients=("PatientID", "nunique"),
    studies=("StudyInstanceUID", "nunique"),
    series=("SeriesInstanceUID", "nunique"),
    images=("ImageCount", "sum"),
    size_mb=("FileSize", "sum"),
)
inventory["size_mb"] = (inventory["size_mb"] / 1e6).round(1)
inventory = inventory.sort_values("images", ascending=False)

total_row = inventory.sum(numeric_only=True)
total_row["patients"] = tcia["PatientID"].nunique()
total_row["studies"] = tcia["StudyInstanceUID"].nunique()
inventory.loc["TOTAL"] = total_row
inventory.to_csv(os.path.join(phase_results, "inventory.csv"))
inventory

,patients,studies,series,images,size_mb
Modality,,,,,
PT,29.0,37.0,48.0,15724.0,646.4
CT,39.0,48.0,49.0,4406.0,2329.3
MR,52.0,55.0,60.0,3511.0,1122.1
US,24.0,25.0,25.0,167.0,1038.6
DX,22.0,25.0,27.0,34.0,452.9
MG,25.0,25.0,25.0,31.0,1645.5
CR,22.0,23.0,24.0,26.0,392.4
SR,21.0,22.0,22.0,22.0,1.7
TOTAL,216.0,241.0,280.0,23921.0,7628.9


## 3. Open every file (header only) to check it is readable

I read files in parallel (16 processes). Only the header is read, so memory use stays small.

In [6]:
file_table = dicom_io.index_folder(paths.VALIDATION_IMAGES, workers=16)

is_dicom_name = file_table["path"].str.endswith(".dcm")
file_table = file_table[is_dicom_name].reset_index(drop=True)

n_files = len(file_table)
n_readable = file_table["readable"].sum()
print("files checked:", n_files)
print("readable     :", n_readable, "(", round(100 * n_readable / n_files, 2), "% )")

unreadable = file_table[~file_table["readable"]]
unreadable[["path", "error"]].head(20)

files checked: 23921
readable     : 23921 ( 100.0 % )


,path,error


In [7]:
# save the full file table (has synthetic IDs, so it stays out of git)
file_table_path = os.path.join(paths.DERIVED, "validation_file_table.csv")
file_table.to_csv(file_table_path, index=False)
print("saved", file_table_path)

saved data/derived/validation_file_table.csv


### Transfer syntaxes, sizes and other quick facts

In [8]:
syntax_names = {
    "1.2.840.10008.1.2": "Implicit VR Little Endian",
    "1.2.840.10008.1.2.1": "Explicit VR Little Endian",
    "1.2.840.10008.1.2.2": "Explicit VR Big Endian",
    "1.2.840.10008.1.2.4.50": "JPEG Baseline",
    "1.2.840.10008.1.2.4.70": "JPEG Lossless",
    "1.2.840.10008.1.2.4.90": "JPEG 2000 Lossless",
    "1.2.840.10008.1.2.4.91": "JPEG 2000",
    "1.2.840.10008.1.2.5": "RLE Lossless",
}
file_table["syntax_name"] = file_table["transfer_syntax"].map(syntax_names)
file_table["syntax_name"] = file_table["syntax_name"].fillna(file_table["transfer_syntax"])

syntax_counts = file_table["syntax_name"].value_counts()
syntax_counts.to_csv(os.path.join(phase_results, "transfer_syntaxes.csv"))
syntax_counts

syntax_name
Explicit VR Little Endian    23901
JPEG 2000 Lossless              20
Name: count, dtype: int64

In [9]:
facts = {}
facts["files"] = n_files
facts["readable_pct"] = round(100 * n_readable / n_files, 2)
facts["files_with_private_tags_pct"] = round(100 * (file_table["n_private_tags"] > 0).mean(), 2)
facts["burned_in_annotation_YES"] = int((file_table["burned_in_annotation"] == "YES").sum())
facts["multi_frame_files"] = int((file_table["frames"].astype(float) > 1).sum())
facts["mean_file_kb"] = round(file_table["size_bytes"].mean() / 1024, 1)
pd.Series(facts)

files                          23921.0
readable_pct                     100.0
files_with_private_tags_pct      100.0
burned_in_annotation_YES           0.0
multi_frame_files                 44.0
mean_file_kb                     311.4
dtype: float64

## 4. Mapping files (old ID -> new ID)

The scorer finds the answer for an output file by mapping its new UID back to the original one.
So my pipeline must give each UID exactly the new value in this table, and each patient the new
Patient ID (e.g. `MIDI_1_1_001`). In a hospital this table would be the pseudonymisation key held by the data custodian.

In [10]:
patient_map = pd.read_csv(paths.VALIDATION_PATIENT_MAP, dtype=str)
uid_map = pd.read_csv(paths.VALIDATION_UID_MAP, dtype=str)
print("patient map rows:", len(patient_map))
print("uid map rows    :", len(uid_map))

patients_in_data = set(file_table["patient_id"])
patients_in_map = set(patient_map["id_old"])
missing_patients = patients_in_data - patients_in_map
print("patients in data not in map:", len(missing_patients))

uids_in_map = set(uid_map["id_old"])
mapping_checks = {}
for column in ["study_uid", "series_uid", "sop_uid"]:
    uids_in_data = set(file_table[column])
    missing = uids_in_data - uids_in_map
    mapping_checks[column] = {"in_data": len(uids_in_data), "missing_from_map": len(missing)}
pd.DataFrame(mapping_checks)

patient map rows: 216
uid map rows    : 27946
patients in data not in map: 0


,study_uid,series_uid,sop_uid
in_data,241,280,23921
missing_from_map,0,0,0


## 5. Answer key

The answer key is a SQLite database with one table `answer_data`. Each row is one DICOM instance
(or series / study) and holds a JSON list of required checks (e.g. `<text_removed>` for a tag).

In [11]:
answer_files = glob.glob(os.path.join(paths.VALIDATION_ANSWER_DIR, "**", "*.db"), recursive=True)
print("answer key files:", answer_files)

if len(answer_files) > 0:
    connection = sqlite3.connect(answer_files[0])
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", connection)
    print(tables)
    answer_df = pd.read_sql("SELECT * FROM answer_data", connection)
    connection.close()
    print("answer rows:", len(answer_df))
    print("columns    :", list(answer_df.columns))

    sops_in_data = set(file_table["sop_uid"])
    sops_in_key = set(answer_df["SOPInstanceUID"])
    linked = len(sops_in_key & sops_in_data)
    print("answer-key instances linked to a file:", linked, "/", len(sops_in_key))
else:
    print("Answer key not downloaded yet -> see docs/DATA.md")

answer key files: ['data/midi_b/validation/answer_key/MIDI-B-Answer-Key-Validation.db']
          name
0  answer_data


answer rows: 23921
columns    : ['index', 'group', 'Modality', 'SOPClassUID', 'PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID', 'Digest', 'AnswerData']
answer-key instances linked to a file: 23921 / 23921


## 6. dev-mini: a small subset for fast experiments

20 patients, chosen so that every modality appears. Fixed random seed = repeatable.

In [12]:
rng = np.random.default_rng(42)

patient_modality = tcia.groupby("PatientID")["Modality"].first()
modalities = sorted(patient_modality.unique())

dev_mini = []
per_modality = 20 // len(modalities)
for modality in modalities:
    candidates = patient_modality[patient_modality == modality].index.tolist()
    candidates.sort()
    n_pick = min(per_modality, len(candidates))
    picked = rng.choice(candidates, size=n_pick, replace=False)
    dev_mini.extend(picked.tolist())

others = [p for p in sorted(patient_modality.index) if p not in dev_mini]
n_missing = 20 - len(dev_mini)
extra = rng.choice(others, size=n_missing, replace=False)
dev_mini.extend(extra.tolist())

dev_mini_path = os.path.join(paths.DERIVED, "dev_mini_patients.txt")
with open(dev_mini_path, "w") as f:
    f.write("\n".join(dev_mini))

dev_files = file_table[file_table["patient_id"].isin(dev_mini)]
print("dev-mini patients:", len(dev_mini), "files:", len(dev_files))
dev_files["modality"].value_counts()

dev-mini patients: 20 files: 1192


modality
CT    779
MR    214
PT    185
MG      4
DX      3
SR      3
US      2
CR      2
Name: count, dtype: int64

## Phase 0 summary

In [13]:
summary = {}
summary["files expected"] = int(expected)
summary["files found"] = int(found)
summary["completeness %"] = round(100 * found / expected, 2)
summary["readable %"] = facts["readable_pct"]
summary["patients missing in patient map"] = len(missing_patients)
summary["answer key present"] = len(answer_files) > 0
summary_table = pd.Series(summary)
summary_table.to_csv(os.path.join(phase_results, "phase0_summary.csv"))
summary_table

files expected                     23921
files found                        23921
completeness %                     100.0
readable %                         100.0
patients missing in patient map        0
answer key present                  True
dtype: object